# Kapitel 19.3 - CGI Praxis, Projekte und Sicherheit

Dieses Notebook ist bewusst sehr praxisnah aufgebaut.
Wir kombinieren CGI-Logik mit sauberer Struktur, Fehlermanagement und Sicherheitsdenken.

# Lernziele

Nach diesem Notebook kannst du:

- kleine CGI-Projekte modular aufbauen
- Eingaben normalisieren und validieren
- HTML-Ausgaben gegen einfache Injection-Risiken absichern
- Fehlerfaelle nutzerfreundlich behandeln
- sauberen und didaktisch nachvollziehbaren Code schreiben

# Voraussetzungen

Empfohlen:

- Notebook 19.1 und 19.2
- Kapitel 10 (Fehlerbehandlung)
- Kapitel 13 (Textverarbeitung)

Zusaetzlich hilfreich: Grundwissen zu Dictionaries und Funktionen mit Rueckgabewerten.

# Theorie

## Sicherheitsgrundsatz

Alle Eingaben aus dem Web sind untrusted input.
Das gilt auch dann, wenn das Formular scheinbar nur intern genutzt wird.

## Warum Escaping wichtig ist

Wenn Benutzereingaben ungefiltert in HTML eingefuegt werden, kann unerwuenschter Code angezeigt oder ausgefuehrt werden.
Darum muessen Sonderzeichen wie `<`, `>`, `&`, `"` sicher kodiert werden.

## Robustheit

Eine gute Webfunktion liefert auch bei Fehlern eine nachvollziehbare Antwort statt eines Absturzes.

# Erklaerung

Wir bauen die Anwendung in klaren Schichten:

1. Parsing (Eingaben lesen)
2. Normalisierung (Whitespace, Klein/Grossschreibung, Typen)
3. Validierung (Regeln pruefen)
4. Antwortaufbau (HTML)

Diese Trennung reduziert Fehler und erleichtert Wartung.

# Syntax

## HTML-Escaping in Python

```python
from html import escape
safe_text = escape(user_input)
```

## Fehlerbehandlungsmuster

```python
try:
    ...
except ValueError:
    ...
```

# Merke

- Eingaben nie direkt in HTML einsetzen.
- Fehlerbehandlung ist Teil der fachlichen Logik.
- Kleine, klare Funktionen schlagen grosse Monolithen.

# Parameter

In den folgenden Funktionen verwenden wir u. a.:

- `query_string`: Rohdaten aus URL oder Formular
- `pflichtfelder`: Liste notwendiger Felder
- `max_laenge`: Begrenzung fuer Textfelder

Gerade solche Parameter machen die Funktionen wiederverwendbar.

# Rueckgabewert

Viele Funktionen geben ein Tupel zurueck:

- bereinigte Daten
- Fehlerliste

Dieses Muster ist in Validierungs-Pipelines sehr praktisch.

In [ ]:
# Beispiel 1: Eingaben sicher escapen
from html import escape

unsicher = '<script>alert(1)</script> Max & Moritz'
sicher = escape(unsicher)

print('Original:', unsicher)
print('Escaped :', sicher)

# Beispiel 1 - Erklaerung

Durch `escape` werden kritische Zeichen in sichere HTML-Entities umgewandelt.
Der Text bleibt sichtbar, aber wird nicht als HTML/Script interpretiert.

In [ ]:
# Beispiel 2: Validierungspipeline fuer ein Kontaktformular
from urllib.parse import parse_qs

def parse_und_normalisiere(query_string):
    roh = parse_qs(query_string, keep_blank_values=True)
    daten = {
        'name': roh.get('name', [''])[0].strip(),
        'email': roh.get('email', [''])[0].strip().lower(),
        'nachricht': roh.get('nachricht', [''])[0].strip()
    }
    return daten

def validiere_kontakt(daten, max_laenge=300):
    fehler = []

    if not daten['name']:
        fehler.append('Name ist erforderlich.')

    if '@' not in daten['email'] or '.' not in daten['email']:
        fehler.append('E-Mail ist ungueltig.')

    if not daten['nachricht']:
        fehler.append('Nachricht darf nicht leer sein.')
    elif len(daten['nachricht']) > max_laenge:
        fehler.append(f'Nachricht darf maximal {max_laenge} Zeichen haben.')

    return fehler

testdaten = parse_und_normalisiere('name=Zeynep&email=zeynep%40mail.de&nachricht=Hallo')
print(testdaten)
print(validiere_kontakt(testdaten))

# Beispiel 2 - Erklaerung

Die Pipeline trennt Parsing und Validierung sauber.
Dadurch bleibt jede Funktion klein und testbar.
Diese Struktur kann spaeter fast unveraendert in Framework-Code uebernommen werden.

In [ ]:
# Beispiel 3: Vollstaendige CGI-Antwort mit Fehler-/Erfolgsfall
from html import escape

def render_kontakt_response(query_string):
    daten = parse_und_normalisiere(query_string)
    fehler = validiere_kontakt(daten)

    if fehler:
        fehler_html = ''.join(f'<li>{escape(f)}</li>' for f in fehler)
        body = f'<h1>Kontaktformular - Fehler</h1><ul>{fehler_html}</ul>'
    else:
        name = escape(daten['name'])
        mail = escape(daten['email'])
        nachricht = escape(daten['nachricht'])
        body = (
            '<h1>Nachricht empfangen</h1>'
            f'<p><strong>Name:</strong> {name}</p>'
            f'<p><strong>E-Mail:</strong> {mail}</p>'
            f'<p><strong>Text:</strong> {nachricht}</p>'
        )

    header = 'Content-Type: text/html; charset=utf-8\n\n'
    return header + f'<html><body>{body}</body></html>'

print(render_kontakt_response('name=Ali&email=ali%40mail.de&nachricht=Ich%20habe%20eine%20Frage')[:220] + '...')
print(render_kontakt_response('name=&email=falsch&nachricht=')[:220] + '...')

# Beispiel 3 - Erklaerung

Die Funktion ist ein kleines End-to-End-Beispiel:
Eingabe -> Normalisierung -> Validierung -> sichere HTML-Ausgabe.
Genau dieses Denkmuster ist spaeter in realen Projekten entscheidend.

# Praxisbeispiel

Mini-Projekt: Beschwerdeformular mit Prioritaet

Regeln:
- Name Pflicht
- Prioritaet muss `niedrig`, `mittel` oder `hoch` sein
- Text mindestens 10 Zeichen

In [ ]:
def bearbeite_beschwerde(query_string):
    daten = parse_qs(query_string, keep_blank_values=True)
    name = daten.get('name', [''])[0].strip()
    prioritaet = daten.get('prioritaet', [''])[0].strip().lower()
    text = daten.get('text', [''])[0].strip()

    fehler = []
    if not name:
        fehler.append('Name fehlt.')
    if prioritaet not in {'niedrig', 'mittel', 'hoch'}:
        fehler.append('Prioritaet ungueltig.')
    if len(text) < 10:
        fehler.append('Beschwerdetext ist zu kurz.')

    if fehler:
        body = '<h1>Beschwerde nicht gespeichert</h1><ul>' + ''.join(f'<li>{escape(f)}</li>' for f in fehler) + '</ul>'
    else:
        body = f'<h1>Beschwerde gespeichert</h1><p>Prioritaet: {escape(prioritaet)}</p><p>Danke, {escape(name)}.</p>'

    return 'Content-Type: text/html; charset=utf-8\n\n' + f'<html><body>{body}</body></html>'

print(bearbeite_beschwerde('name=Arda&prioritaet=hoch&text=Der%20Login%20funktioniert%20nicht')[:200] + '...')
print(bearbeite_beschwerde('name=&prioritaet=extrem&text=kurz')[:200] + '...')

# Haeufige Fehler

1. Unsichere direkte HTML-Ausgabe ohne `escape`.
2. Fehlertexte aus Ausnahmen ungefiltert an den Benutzer senden.
3. Zu spaete Validierung (erst nach vielen Verarbeitungsschritten).
4. Keine Begrenzung fuer Eingabelaengen.
5. Fehlende Standardwerte bei optionalen Feldern.

# Best Practice

- Nutze Whitelists fuer feste Auswahlfelder (z. B. Prioritaet).
- Pruefe Eingaben so frueh wie moeglich.
- Escape alle benutzergesteuerten Texte vor HTML-Ausgabe.
- Definiere klare Fehlermeldungen fuer Nutzer und getrennte Debug-Infos fuer Entwickler.
- Halte Funktionen klein und fokussiert.

# Tipp

Baue dir fuer Lernzwecke eine kleine Helper-Datei mit wiederkehrenden Funktionen:
`parse_input`, `validate_input`, `render_html`, `render_errors`.
So lernst du frueh modulare Architekturprinzipien.

# Uebung

Entwickle ein 'Event-Anmeldung'-Mini-System:

- Felder: `name`, `event`, `plaetze`
- `event` darf nur aus einer festen Liste stammen
- `plaetze` muss Ganzzahl zwischen 1 und 5 sein
- Gib bei Erfolg eine Zusammenfassung, bei Fehlern eine Liste aus

In [ ]:
# Loesung
def event_anmeldung(query_string):
    daten = parse_qs(query_string, keep_blank_values=True)
    name = daten.get('name', [''])[0].strip()
    event = daten.get('event', [''])[0].strip()
    plaetze_text = daten.get('plaetze', [''])[0].strip()

    erlaubte_events = {'Python Basics', 'Web Grundlagen', 'Datenanalyse'}
    fehler = []

    if not name:
        fehler.append('Name fehlt.')

    if event not in erlaubte_events:
        fehler.append('Event ist ungueltig.')

    if not plaetze_text.isdigit():
        fehler.append('Plaetze muss eine Zahl sein.')
    else:
        plaetze = int(plaetze_text)
        if not (1 <= plaetze <= 5):
            fehler.append('Plaetze muss zwischen 1 und 5 liegen.')

    if fehler:
        body = '<h1>Anmeldung fehlgeschlagen</h1><ul>' + ''.join(f'<li>{escape(f)}</li>' for f in fehler) + '</ul>'
    else:
        body = (
            f'<h1>Anmeldung bestaetigt</h1>'
            f'<p>Name: {escape(name)}</p>'
            f'<p>Event: {escape(event)}</p>'
            f'<p>Plaetze: {plaetze_text}</p>'
        )

    return 'Content-Type: text/html; charset=utf-8\n\n' + f'<html><body>{body}</body></html>'

print(event_anmeldung('name=Yusuf&event=Python%20Basics&plaetze=2')[:220] + '...')
print(event_anmeldung('name=&event=Unbekannt&plaetze=99')[:220] + '...')

# Zusammenfassung

Du hast CGI jetzt auf einem professionelleren Niveau geuebt:

- sichere Ausgabe mit Escaping
- modulare Validierungs-Pipelines
- robuste Fehlerbehandlung
- praxisnahe Mini-Projekte

Im naechsten Notebook wechseln wir auf WSGI und bauen eine dauerhafte Python-Webanwendung.

# Weiterfuehrende Links

- Python `html.escape` Dokumentation
- OWASP XSS Prevention Cheat Sheet
- PEP 3333 als Bruecke zu WSGI

## Technischer Tiefgang

Der Fokus liegt auf reproduzierbaren technischen Entscheidungen statt auf isolierten Einzelbeispielen.
Dabei werden Architektur, Robustheit und Betriebsfaehigkeit gemeinsam betrachtet.

## Zentrale Fachbegriffe

HTTP Semantics
Status Code Family
WSGI Callable
Request Lifecycle
Input Sanitization
Header Validation

In [ ]:
# WSGI-Minibeispiel mit Statuscode
def app(environ, start_response):
    path = environ.get("PATH_INFO", "/")
    if path == "/health":
        start_response("200 OK", [("Content-Type", "text/plain")])
        return [b"ok"]
    start_response("404 Not Found", [("Content-Type", "text/plain")])
    return [b"not found"]

## Fallstudie (Praxis)

Waehle ein realistisches Produktionsszenario und beschreibe systematisch Ursache, Risiko und technische Gegenmassnahmen.
Ergaenze mindestens ein Kriterium fuer Monitoring und ein Kriterium fuer Release-Entscheidungen.

## Haeufige Fehler und Debugging-Checkliste

- Ist das Problem reproduzierbar mit klaren Schritten?
- Sind relevante Signale vorhanden (Logs, Tests, Metriken)?
- Wurde eine konkrete Hypothese getestet und falsifiziert/bestaetigt?
- Ist die Korrektur durch einen Regressionstest abgesichert?
- Wurden Betriebsfolgen und Dokumentation mit aktualisiert?

## Pruefungsfragen und Kurzloesungen

1. Warum ist Reproduzierbarkeit in Fehleranalyse und Betrieb zentral?
Kurzloesung: Ohne reproduzierbare Befunde sind Ursachenanalyse, Fix und Absicherung nicht belastbar.
2. Was unterscheidet technische Begriffe von bloessem Buzzword-Einsatz?
Kurzloesung: Praezise Begriffe steuern messbare Entscheidungen und verbessern Teamkommunikation.
3. Welche Mindestkriterien sollte ein Release-Gate enthalten?
Kurzloesung: Teststatus, Sicherheitschecks, Fehlerbudget und nachvollziehbare Freigabeentscheidung.